In [26]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [27]:
spark = (
    SparkSession.builder
    .appName("OlistSilverLayer")\
    .master("local[*]")\
    .getOrCreate()
)

print("Spark Version: ",spark.version)

Spark Version:  4.2.0


In [28]:
from pathlib import Path

olist_order_dataset_path = Path.cwd().parent/"data"/"bronze"/"olist_orders_dataset"

bronze_df = (
    spark.read.format('parquet')
    .option('inferSchema',True)
    .option('header',True)\
    .load(str(olist_order_dataset_path))
)

bronze_df.show(5)
bronze_df.printSchema()


+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+--------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date| ingestion_timestamp|         source_file|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+--------------------+
|995392413cee61cc1...|4bf24904ec428325a...|   delivered|     2017-09-04 22:24:05|2017-09-04 22:43:54|         2017-09-13 17:20:04|          2017-09-22 21:09:32|          2017-09-27 00:00:00|2026-10-12 22:47:...|olist_orders_data...|
|b39de9ed2bb8fd08a...|ed18b557140ff674f...|   delivered|     2018-03

In [29]:
columns_to_convert = ['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date','order_delivered_customer_date','order_estimated_delivery_date']

for column in columns_to_convert:
    silver_df = bronze_df.withColumn(column,col(column).cast('timestamp'))

silver_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



In [30]:
print(col)
print(type(col))

<function col at 0x10bc62ca0>
<class 'function'>


In [31]:
#identifying any values that were present in Bronze but became NULL after conversion to a timestamp

joined_df = bronze_df.alias("b").join(silver_df.alias("s"),
               bronze_df["order_id"]==silver_df["order_id"],"inner")

joined_df.show()

26/10/12 23:36:43 WARN Column: Constructing trivially true equals predicate, 'order_id == order_id'. Perhaps you need to use aliases.


+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+--------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date| ingestion_timestamp|         source_file|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date| ingestion_timestamp|         source_file|
+--------------------+--------------------+------------+----------------

In [32]:
joined_df.select(
    col("b.order_id"),
    col("b.order_purchase_timestamp"),
    col("s.order_purchase_timestamp")
).show(10)

+--------------------+------------------------+------------------------+
|            order_id|order_purchase_timestamp|order_purchase_timestamp|
+--------------------+------------------------+------------------------+
|995392413cee61cc1...|     2017-09-04 22:24:05|     2017-09-04 22:24:05|
|b39de9ed2bb8fd08a...|     2018-03-27 09:15:17|     2018-03-27 09:15:17|
|b1a88554eb1f7f686...|     2018-06-13 02:19:56|     2018-06-13 02:19:56|
|fe2ac30768e07e362...|     2017-01-16 16:41:18|     2017-01-16 16:41:18|
|460673777918e7a42...|     2018-02-07 22:38:41|     2018-02-07 22:38:41|
|c170c51c1eb31dee8...|     2017-10-04 15:32:22|     2017-10-04 15:32:22|
|575ca88a26f8e32bb...|     2018-01-10 12:01:27|     2018-01-10 12:01:27|
|20fb6eab29c449df8...|     2018-02-02 10:06:00|     2018-02-02 10:06:00|
|96c21ec9db51ad9d0...|     2017-08-14 20:49:22|     2017-08-14 20:49:22|
|750ed62dff96bf571...|     2018-05-28 11:18:59|     2018-05-28 11:18:59|
+--------------------+------------------------+----

In [33]:
joined_df.filter(
    col("b.order_purchase_timestamp").isNotNull() &
    col("s.order_purchase_timestamp").isNull()
).show()

+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+-------------------+-----------+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+-------------------+-----------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|ingestion_timestamp|source_file|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|ingestion_timestamp|source_file|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+-------------------+-----------+-----

In [35]:
#we compared one timestamp - now lets move to all the columns
for column in columns_to_convert:
    bronze_column = col(f"b.{column}")
    silver_column = col(f"s.{column}")

    failed_df = joined_df.filter(
        bronze_column.isNotNull() & silver_column.isNull()
    )

    print(f"{column}: {failed_df.count()} failed conversions")

order_purchase_timestamp: 0 failed conversions
order_approved_at: 0 failed conversions
order_delivered_carrier_date: 0 failed conversions
order_delivered_customer_date: 0 failed conversions
order_estimated_delivery_date: 0 failed conversions


In [37]:
#efficient method of doing above approach
validation_expr = []

for column in columns_to_convert:
    bronze_column = col(f"b.{column}")
    silver_column = col(f"s.{column}")

    condition = bronze_column.isNotNull() & silver_column.isNull()

    validation_expr.append(
        sum(condition.cast("int")).alias(f"{column}_failures")
    )

joined_df.select(validation_expr).show()

+---------------------------------+--------------------------+-------------------------------------+--------------------------------------+--------------------------------------+
|order_purchase_timestamp_failures|order_approved_at_failures|order_delivered_carrier_date_failures|order_delivered_customer_date_failures|order_estimated_delivery_date_failures|
+---------------------------------+--------------------------+-------------------------------------+--------------------------------------+--------------------------------------+
|                                0|                         0|                                    0|                                     0|                                     0|
+---------------------------------+--------------------------+-------------------------------------+--------------------------------------+--------------------------------------+



In [45]:
condition = silver_df.filter((col('order_id').isNull()) | (trim('order_id') == ""))
#i mean above was my code but see since we are filtering on entire data frame we just do like 
#silver_df.filter((col('order_id').isNull()) | (trim('order_id')=="")).show()
result = condition.count()
print('Number of invalid columns: ',result)

Number of invalid columns:  0


In [52]:
silver_df.groupBy('order_id').count()\
.filter(col('count')>1).show()

+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+

